# OpenToken PySpark Example

This notebook demonstrates how to use the OpenToken PySpark bridge to generate privacy-preserving tokens from a PySpark DataFrame.

## Prerequisites

1. Install the required packages:
   ```bash
   cd lib/python/opentoken
   uv pip install -e .
   cd ../opentoken-pyspark
   uv pip install -e .[spark40,dev]
   ```

2. Ensure you have PySpark and Jupyter installed

## Setup

In [ ]:
# Import required libraries
from pyspark.sql import SparkSession

from opentoken_pyspark import OpenTokenProcessor

In [ ]:
# Create a Spark session
spark = (
    SparkSession.builder.appName("OpenTokenExample")
    .master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

print(f"Spark version: {spark.version}")

## Load Sample Data

We'll load the sample CSV data into a PySpark DataFrame.

In [ ]:
# Load sample data from CSV
sample_csv_path = "../../../../resources/sample.csv"

df = spark.read.csv(sample_csv_path, header=True, inferSchema=True)

# Display the schema
print("Input DataFrame Schema:")
df.printSchema()

# Show first few rows
print("\nFirst 5 rows:")
df.show(5, truncate=False)

## Initialize OpenToken Processor

Create an instance of `OpenTokenProcessor` from your initiate-exchange config.

**Recommended:** Resolve the exchange config and participant private key on the driver with `from_exchange_config(...)`.
**Backward compatibility:** Direct `hashing_secret` / `encryption_key` construction still works, but it is no longer the recommended primary setup flow.


In [ ]:
# Initialize the processor from exchange-config inputs
processor = OpenTokenProcessor.from_exchange_config(
    exchange_config_path="/path/to/initiate-exchange-config.json",
    private_key_path="/path/to/participant-private-key.pem",
)

print("OpenToken Processor initialized successfully!")

### Azure Key Vault Pattern

When your Spark job runs in Azure, resolve Key Vault secrets on the **driver** before calling `OpenTokenProcessor.from_exchange_config(...)`. The bridge needs the initiate-exchange config JSON plus the participant private key; the sender and recipient public keys are already embedded in the generated exchange config payload.

If Azure Key Vault is your system of record, fetch the exchange-config JSON and participant private key from Key Vault, then pass those values directly into the PySpark bridge:

```python
import os

from azure.identity import DefaultAzureCredential
from azure.keyvault.secrets import SecretClient
from opentoken_pyspark import OpenTokenProcessor

vault = SecretClient(
    vault_url=os.environ["AZURE_KEY_VAULT_URL"],
    credential=DefaultAzureCredential(),
)

exchange_config_json = vault.get_secret("opentoken-initiate-exchange-config").value
participant_private_key_pem = vault.get_secret("opentoken-participant-private-key-pem").value

# Optional: keep the public keys in Key Vault for exchange creation, rotation, or validation.
sender_public_key_pem = vault.get_secret("opentoken-sender-public-key-pem").value
recipient_public_key_pem = vault.get_secret("opentoken-recipient-public-key-pem").value

processor = OpenTokenProcessor.from_exchange_config(
    exchange_config_value=exchange_config_json,
    private_key_value=participant_private_key_pem,
)
```

If you store the public keys separately in Key Vault, use them when you create or rotate the initiate-exchange config upstream. The PySpark bridge does not take public-key arguments directly.


## Generate Tokens

Process the DataFrame to generate tokens for each person record.

In [ ]:
# Generate tokens
tokens_df = processor.process_dataframe(df)

# Display the schema of the result
print("Output DataFrame Schema:")
tokens_df.printSchema()

# Count total tokens generated
total_tokens = tokens_df.count()
print(f"\nTotal tokens generated: {total_tokens}")

## Inspect Results

Let's look at the generated tokens for a specific record.

In [ ]:
# Show tokens for the first record
first_record_id = df.select("RecordId").first()[0]
print(f"Tokens for RecordId: {first_record_id}")

tokens_df.filter(tokens_df.RecordId == first_record_id).show(truncate=False)

## Analyze Token Distribution

Check how many tokens were generated per rule.

In [ ]:
# Count tokens by RuleId
print("Token count by RuleId:")
tokens_df.groupBy("RuleId").count().orderBy("RuleId").show()

## Convert to Pandas for Visualization

For smaller datasets, you can convert to Pandas for easier visualization.

In [ ]:
# Convert a subset to Pandas for visualization
sample_tokens = tokens_df.limit(10).toPandas()
print("Sample tokens as Pandas DataFrame:")
display(sample_tokens)

## Save Results

Save the tokens to a Parquet file for further processing.

In [ ]:
# Save to Parquet
output_path = "../output/tokens_output.parquet"

tokens_df.write.mode("overwrite").parquet(output_path)
print(f"Tokens saved to: {output_path}")

## Example: Using Alternative Column Names

OpenToken supports alternative column names for flexibility.

In [ ]:
# Create a DataFrame with alternative column names
alt_data = [
    {
        "Id": "custom-001",
        "GivenName": "Alice",
        "Surname": "Johnson",
        "ZipCode": "98052",
        "Gender": "Female",
        "DateOfBirth": "1990-05-15",
        "NationalIdentificationNumber": "234-56-7890",
    }
]

alt_df = spark.createDataFrame(alt_data)

# Process with alternative column names
alt_tokens_df = processor.process_dataframe(alt_df)

print("Tokens generated with alternative column names:")
alt_tokens_df.show(truncate=False)

## Cleanup

Stop the Spark session when done.

In [ ]:
# Stop Spark session
spark.stop()

## Summary

This notebook demonstrated:

1. Loading person data into a PySpark DataFrame
2. Initializing the OpenToken processor with secrets
3. Generating privacy-preserving tokens for each record
4. Analyzing and visualizing the results
5. Saving tokens for further use

The PySpark bridge enables distributed token generation for large-scale person matching workflows.